# 05. Testes de Carga, Validação de Latência e Telemetria

**Objetivo Científico:** Comprovar experimentalmente que a **Arquitetura Medallion** implementada no Databricks elimina os gargalos de latência analítica no processamento do acervo OpenAlex.

Neste experimento de benchmarking, comparamos a recuperação de redes de coautoria sob dois paradigmas arquiteturais distintos:
1. **Abordagem Tradicional em Data Lake (Camada Bronze):** Consulta analítica direta sobre arquivos JSON Lines semiestruturados e aninhados;
2. **Abordagem Otimizada em Lakehouse (Camada Ouro):** Consulta relacional sobre tabelas Delta colunares modeladas sob a Teoria dos Grafos (`gold_arestas` e `gold_vertices`).

---
### Pesquisador Alvo do Teste
* **Nome:** Dr. Eric J. Topol
* **Identificador OpenAlex:** `https://openalex.org/A5084515381`
* **Contexto de Domínio:** Médico e cientista líder global na aplicação de Inteligência Artificial na Medicina (autor de *Deep Medicine*), possuindo uma rede densa de publicações e coautorias interdisciplinares no dataset.

In [0]:
import time
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql.functions import col, explode

print("=" * 30 + " AMBIENTE DE TELEMETRIA " + "=" * 30)
TARGET_AUTHOR = "https://openalex.org/A5084515381"
TARGET_AUTHOR_NAME = "Dr. Eric J. Topol"

print(f"Pesquisador Alvo Selecionado: {TARGET_AUTHOR_NAME}")
print(f"Identificador OpenAlex:      {TARGET_AUTHOR}")
print("Objetivo: Recuperar todos os coautores diretos que colaboraram em artigos científicos com o alvo.")
print("=" * 84)

# Limpa o cache para garantir medição estrita de I/O em disco
spark.catalog.clearCache()

## 1. Teste de Linha de Base: Camada Bronze (JSON Aninhado)

### Mecânica Interna da Busca no JSON:
* **Varredura Completa (Full Table Scan):** O Spark precisa ler todas as linhas do JSON bruto e desserializar estruturas em árvore na memória RAM;
* **Função de Alta Ordem em Matriz:** Para verificar se o autor escreveu a obra, avalia a expressão `exists(authorships, a -> a.author.id == TARGET_AUTHOR)` linha a linha em tempo de execução;
* **Desaninhamento Tardio (`explode`):** Desmonta a matriz `authorships` de cada artigo encontrado para projetar cada coautor individualmente, exigindo `dropDuplicates()` com alto custo de *shuffle* distribuído.
* **Gargalo:** Alto consumo de CPU, pressão de Garbage Collection (GC) e impossibilidade de aplicar poda colunar.

In [0]:
print("[1] Iniciando consulta analítica na Camada Bronze (JSON Aninhado)...")
inicio_bronze = time.time()

df_bronze = spark.table("bronze_openalex")

# 1. Filtra artigos onde o autor alvo está presente dentro do array aninhado
df_bronze_filtered = df_bronze.filter(
    f"exists(authorships, a -> a.author.id == '{TARGET_AUTHOR}')"
)

# 2. Desaninha o array de autores dos artigos identificados
df_bronze_coautores = df_bronze_filtered.select(explode("authorships").alias("auth")) \
    .select(col("auth.author.id").alias("coautor_id"), col("auth.author.display_name").alias("nome")) \
    .filter(col("coautor_id") != TARGET_AUTHOR) \
    .dropDuplicates(["coautor_id"])

# Força a execução física da DAG e computa a telemetria de latência
count_bronze = df_bronze_coautores.count()
fim_bronze = time.time()
tempo_bronze = fim_bronze - inicio_bronze

print(f"-> Sucesso Bronze: {count_bronze} coautores únicos localizados.")
print(f"-> Tempo de Execução Bronze: {tempo_bronze:.4f} segundos")

## 2. Teste Otimizado: Camada Ouro (Modelagem de Grafos em Delta Lake)

### Mecânica Interna da Busca no Modelo de Grafos:
* **Poda Colunar e Predicate Pushdown:** O Spark lê apenas as colunas `id_origem`, `id_destino` e `tipo_aresta` do formato Parquet/Delta, ignorando gigabytes de texto e metadados;
* **Estatísticas Min/Max nos Metadados Delta:** O leitor salta blocos de dados inteiros que não contêm o ID do autor alvo;
* **Junção Relacional Direta (Inner Join):** Cruza a aresta de autoria do autor alvo diretamente com a tabela `gold_arestas` para localizar coautores e resolve o nome com um lookup atômico em `gold_vertices`;
* **Ganho:** Zero desaninhamento de arrays em tempo de consulta, viabilizando consumo web em milissegundos.

In [0]:
print("[2] Iniciando consulta analítica na Camada Ouro (Delta/Grafos Colunares)...")
inicio_ouro = time.time()

df_arestas = spark.table("gold_arestas")
df_vertices = spark.table("gold_vertices")

# 1. Recupera as arestas de autoria ligadas ao autor alvo (obras escritas por ele)
df_artigos_alvo = df_arestas.filter(
    (col("tipo_aresta") == "Autoria") & (col("id_origem") == TARGET_AUTHOR)
).select(col("id_destino").alias("work_id"))

# 2. Junção com as demais arestas de autoria das mesmas obras para identificar coautores
df_coautorias = df_arestas.filter(col("tipo_aresta") == "Autoria") \
    .join(df_artigos_alvo, df_arestas.id_destino == df_artigos_alvo.work_id, "inner") \
    .filter(col("id_origem") != TARGET_AUTHOR) \
    .select(col("id_origem").alias("coautor_id")) \
    .dropDuplicates(["coautor_id"])

# 3. Lookup final nos vértices apenas para recuperar a entidade nominal do pesquisador
df_ouro_coautores = df_coautorias.join(df_vertices, df_coautorias.coautor_id == df_vertices.id_no, "inner") \
    .select(col("id_no"), col("nome_no"))

count_ouro = df_ouro_coautores.count()
fim_ouro = time.time()
tempo_ouro = fim_ouro - inicio_ouro

print(f"-> Sucesso Ouro: {count_ouro} coautores únicos localizados.")
print(f"-> Tempo de Execução Ouro: {tempo_ouro:.4f} segundos")

## 3. Síntese Comparativa e Geração dos Gráficos de Telemetria

A célula abaixo consolida as métricas experimentais de latência e gera a figura comparativa em alta resolução pronta para o **Capítulo 4 (Resultados e Discussões)** do TCC.

### Métricas Calculadas:
* **Speedup ($S$):** $S = \frac{T_{Bronze}}{T_{Ouro}}$ (Fator multiplicador de ganho de velocidade);
* **Redução de Latência ($R$):** $R = \left(1 - \frac{T_{Ouro}}{T_{Bronze}}\right) \times 100\%$ (Percentual de tempo computacional economizado).

In [0]:
# 3. CONSOLIDAÇÃO E VISUALIZAÇÃO GRÁFICA DE DESEMPENHO
speedup = tempo_bronze / tempo_ouro if tempo_ouro > 0 else 0
reducao_latencia = ((tempo_bronze - tempo_ouro) / tempo_bronze) * 100 if tempo_bronze > 0 else 0

print(f"\n{'='*25} SÍNTESE DE DESEMPENHO {'='*25}")
print(f"Tempo Camada Bronze (JSON):  {tempo_bronze:.4f} s")
print(f"Tempo Camada Ouro (Delta):   {tempo_ouro:.4f} s")
print(f"Fator de Aceleração (Speedup): {speedup:.1f}x mais rápido")
print(f"Redução Relativa de Latência: {reducao_latencia:.2f}%")
print("="*72)

# Configuração visual profissional para monografia científica
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
cores = ["#d9534f", "#2e7d32"]  # Vermelho para Bronze (Gargalo), Verde para Ouro (Otimizado)
camadas = ["Bronze\n(JSON Aninhado)", "Ouro\n(Delta Grafos)"]
tempos = [tempo_bronze, tempo_ouro]

# Subplot 1: Escala Linear (Tempo Absoluto de Execução)
bars1 = ax1.bar(camadas, tempos, color=cores, width=0.45, edgecolor="#111", linewidth=1.2, zorder=3)
ax1.set_ylabel("Tempo de Execução (segundos)", fontsize=11, fontweight="bold")
ax1.set_title("Comparação de Latência de Leitura Analítica\n(Escala Linear)", fontsize=12, fontweight="bold", pad=12)
ax1.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)

for bar in bars1:
    h = bar.get_height()
    ax1.annotate(f"{h:.2f} s",
                 xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 6), textcoords="offset points",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

# Subplot 2: Escala Logarítmica (Ordens de Magnitude e Speedup)
bars2 = ax2.bar(camadas, tempos, color=cores, width=0.45, edgecolor="#111", linewidth=1.2, zorder=3)
ax2.set_yscale("log")
ax2.set_ylabel("Tempo em Escala Logarítmica (log10 segundos)", fontsize=11, fontweight="bold")
ax2.set_title("Discrepância em Ordens de Magnitude\n(Escala Logarítmica)", fontsize=12, fontweight="bold", pad=12)
ax2.grid(axis="y", which="both", linestyle="--", alpha=0.5, zorder=0)

for bar in bars2:
    h = bar.get_height()
    ax2.annotate(f"{h:.4f} s",
                 xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 6), textcoords="offset points",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

# Destaque de métrica de aceleração no gráfico
fig.suptitle(f"Benchmark de Desempenho Medallion • Pesquisador Alvo: {TARGET_AUTHOR_NAME}",
             fontsize=14, fontweight="bold", y=1.03)

ax1.text(0.5, 0.75, f"Speedup: ~{speedup:.0f}x\nRedução: {reducao_latencia:.1f}%",
         transform=ax1.transAxes, ha="center", va="center",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#fffde7", edgecolor="#fbc02d", lw=1.5),
         fontsize=10, fontweight="bold", color="#333")

plt.tight_layout()

# Exportação em alta resolução (300 DPI) para o texto do TCC
output_fig = "documentacao_tcc/grafico_performance_benchmark_medallion.png"
try:
    plt.savefig(output_fig, dpi=300, bbox_inches="tight")
    print(f"-> Gráfico salvo com sucesso para o TCC em: {output_fig}")
except Exception as e:
    print(f"Nota: Não foi possível salvar arquivo localmente: {e}")

plt.show()